In [6]:
import pandas as pd
import pyodbc

# Connect to the data warehouse

first_name = "sophie"  # Fill in lowercase
last_name = "wilson"  # Fill in lowercase
password = ""  # Leave blank

conn_str = "DRIVER={ODBC Driver 17 for SQL Server};"
conn_str += "SERVER=dataportal.dwh.frontiersin.net,11433;"
conn_str += "DATABASE=TenantsDataMarts;"
conn_str += "UID=OFFICE\\{}.{};".format(first_name, last_name)
conn_str += "PWD={};".format(password)
conn_str += "Trusted_Connection=yes"


conn_dwh = pyodbc.connect(conn_str)

In [ ]:
# Extract DWH schema for Cursor skill# Extract foreign key relationships
databases = ["TenantsDataMarts", "ReportingDataMart", "FrontiersReports"]


all_schemas = []


for db in databases:

    query = f"""
    SELECT 


        '{db}' AS DATABASE_NAME,


        t.TABLE_SCHEMA,


        t.TABLE_NAME,


        c.COLUMN_NAME,


        c.DATA_TYPE,


        c.IS_NULLABLE,


        c.CHARACTER_MAXIMUM_LENGTH,


        c.NUMERIC_PRECISION


    FROM [{db}].INFORMATION_SCHEMA.TABLES t


    JOIN [{db}].INFORMATION_SCHEMA.COLUMNS c 


        ON t.TABLE_SCHEMA = c.TABLE_SCHEMA 


        AND t.TABLE_NAME = c.TABLE_NAME


    WHERE t.TABLE_TYPE = 'BASE TABLE'


    ORDER BY t.TABLE_SCHEMA, t.TABLE_NAME, c.ORDINAL_POSITION
    """

    df = pd.read_sql_query(query, conn_dwh)

    all_schemas.append(df)


df_schema = pd.concat(all_schemas, ignore_index=True)


# Format as markdown


def format_schema_markdown(df):

    lines = ["# DWH Schema Reference\n"]
    lines.append(
        f"Generated from SQL Server. {len(df)} columns across {df['TABLE_NAME'].nunique()} tables.\n"
    )

    for (db, schema), schema_df in df.groupby(["DATABASE_NAME", "TABLE_SCHEMA"]):

        lines.append(f"\n## [{db}].[{schema}]\n")

        for table, table_df in schema_df.groupby("TABLE_NAME"):

            lines.append(f"\n### {table}\n")

            lines.append("| Column | Type | Nullable |")

            lines.append("|---|---|---|")

            for _, row in table_df.iterrows():

                dtype = row["DATA_TYPE"]
                if (
                    pd.notna(row["CHARACTER_MAXIMUM_LENGTH"])
                    and row["CHARACTER_MAXIMUM_LENGTH"] > 0
                ):

                    dtype += f"({int(row['CHARACTER_MAXIMUM_LENGTH'])})"

                elif pd.notna(row["NUMERIC_PRECISION"]):

                    dtype += f"({int(row['NUMERIC_PRECISION'])})"

                nullable = "YES" if row["IS_NULLABLE"] == "YES" else "NO"

                lines.append(f"| `{row['COLUMN_NAME']}` | {dtype} | {nullable} |")

    return "\n".join(lines)


md_content = format_schema_markdown(df_schema)


with open("../schema-reference.md", "w", encoding="utf-8") as f:

    f.write(md_content)

print(
    f"Schema exported: {len(df_schema)} columns from {df_schema['TABLE_NAME'].nunique()} tables"
)

C:\Users\sophie.wilson\AppData\Local\Temp\ipykernel_40452\1960474662.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn_dwh)


Schema exported: 12338 columns from 565 tables


In [ ]:
all_fks = []

for db in databases:
    fk_query = f"""
    SELECT 
        '{db}' AS DATABASE_NAME,
        cu.CONSTRAINT_NAME AS ForeignKeyName,
        cu.TABLE_SCHEMA AS ReferencingTableSchema,
        cu.TABLE_NAME AS ReferencingTableName,
        cu.COLUMN_NAME AS ReferencingColumnName,
        tc.TABLE_SCHEMA AS ReferencedTableSchema,
        tc.TABLE_NAME AS ReferencedTableName,
        tc.CONSTRAINT_TYPE,
        rc.UNIQUE_CONSTRAINT_NAME AS ReferencedKeyName
    FROM [{db}].INFORMATION_SCHEMA.KEY_COLUMN_USAGE cu
    INNER JOIN [{db}].INFORMATION_SCHEMA.REFERENTIAL_CONSTRAINTS rc 
        ON rc.CONSTRAINT_NAME = cu.CONSTRAINT_NAME
    INNER JOIN [{db}].INFORMATION_SCHEMA.TABLE_CONSTRAINTS tc 
        ON tc.CONSTRAINT_NAME = rc.UNIQUE_CONSTRAINT_NAME
        AND tc.TABLE_SCHEMA = rc.UNIQUE_CONSTRAINT_SCHEMA
    ORDER BY cu.TABLE_SCHEMA, cu.TABLE_NAME, cu.CONSTRAINT_NAME, cu.COLUMN_NAME
    """
    df_fk = pd.read_sql_query(fk_query, conn_dwh)
    all_fks.append(df_fk)

df_relationships = pd.concat(all_fks, ignore_index=True)
print(f"Found {len(df_relationships)} foreign key relationships")
df_relationships.head(10)

C:\Users\sophie.wilson\AppData\Local\Temp\ipykernel_40452\1227704121.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_fk = pd.read_sql_query(fk_query, conn_dwh)


Found 933 foreign key relationships


,DATABASE_NAME,ForeignKeyName,ReferencingTableSchema,ReferencingTableName,ReferencingColumnName,ReferencedTableSchema,ReferencedTableName,CONSTRAINT_TYPE,ReferencedKeyName
0,TenantsDataMarts,FK-Accounting.InstitutionalMembership.ArticleF...,Accounting,InstitutionalMembership.ArticleFunding,ArticleId,Journal,Articles,PRIMARY KEY,PK-Journal.Articles
1,TenantsDataMarts,FK-Accounting.InstitutionalMembership.ArticleF...,Accounting,InstitutionalMembership.ArticleFunding,DeclinationReasonId,Accounting,InstitutionalMembership.DeclinationReason,PRIMARY KEY,PK-Accounting.InstitutionalMembership.Declinat...
2,TenantsDataMarts,FK-Accounting.InstitutionalMembership.ArticleF...,Accounting,InstitutionalMembership.ArticleFunding,FundingDecisionId,Accounting,InstitutionalMembership.FundingDecision,PRIMARY KEY,PK-Accounting.InstitutionalMembership.FundingD...
3,TenantsDataMarts,FK-Accounting.InstitutionalMembership.ArticleF...,Accounting,InstitutionalMembership.ArticleFunding,InstitutionId,Accounting,InstitutionalMembership.Institution,PRIMARY KEY,PK-Accounting.InstitutionalMembership.Institution
4,TenantsDataMarts,FK-Accounting.InstitutionalMembership.ArticleF...,Accounting,InstitutionalMembership.ArticleFunding,InvoiceId,Accounting,Invoices_History,PRIMARY KEY,PK-Accounting.Invoices_History
5,TenantsDataMarts,FK-Accounting.InstitutionalMembership.ArticleF...,Accounting,InstitutionalMembership.ArticleFunding,MatchingReasonId,Accounting,InstitutionalMembership.MatchingReason,PRIMARY KEY,PK-Accounting.InstitutionalMembership.Matching...
6,TenantsDataMarts,FK-Accounting.InstitutionalMembership.ArticleF...,Accounting,InstitutionalMembership.ArticleFunding,SpaceId,Common,Spaces,PRIMARY KEY,PK-Common.Spaces
7,TenantsDataMarts,FK-Accounting.InstitutionalMembership.ArticleF...,Accounting,InstitutionalMembership.ArticleFundingHistory,ArticleId,Journal,Articles,PRIMARY KEY,PK-Journal.Articles
8,TenantsDataMarts,FK-Accounting.InstitutionalMembership.ArticleF...,Accounting,InstitutionalMembership.ArticleFundingHistory,CreatorUserId,Network,Users,PRIMARY KEY,PK-Network.Users
9,TenantsDataMarts,FK-Accounting.InstitutionalMembership.ArticleF...,Accounting,InstitutionalMembership.ArticleFundingHistory,DeclinationReasonId,Accounting,InstitutionalMembership.DeclinationReason,PRIMARY KEY,PK-Accounting.InstitutionalMembership.Declinat...


In [9]:
# Format relationships as markdown and append to schema file
def format_relationships_markdown(df):
    if df.empty:
        return "\n\n---\n\n## Foreign Key Relationships\n\nNo foreign key relationships found.\n"

    lines = ["\n\n---\n\n## Foreign Key Relationships\n"]
    lines.append(
        f"Found {len(df)} foreign key relationships across {df['DATABASE_NAME'].nunique()} databases.\n"
    )

    for db, db_df in df.groupby("DATABASE_NAME"):
        lines.append(f"\n### [{db}] Relationships\n")
        lines.append("| FK Name | From Table | From Column | To Table | To Key |")
        lines.append("|---|---|---|---|---|")

        for _, row in db_df.iterrows():
            from_table = (
                f"[{row['ReferencingTableSchema']}].[{row['ReferencingTableName']}]"
            )
            to_table = (
                f"[{row['ReferencedTableSchema']}].[{row['ReferencedTableName']}]"
            )
            lines.append(
                f"| `{row['ForeignKeyName']}` | {from_table} | `{row['ReferencingColumnName']}` | {to_table} | `{row['ReferencedKeyName']}` |"
            )

    return "\n".join(lines)


# Read existing schema file and append relationships
with open("../schema-reference.md", "r", encoding="utf-8") as f:
    existing_content = f.read()

# Remove old relationships section if present
if "## Foreign Key Relationships" in existing_content:
    existing_content = existing_content.split("---\n\n## Foreign Key Relationships")[
        0
    ].rstrip()

# Append new relationships
fk_markdown = format_relationships_markdown(df_relationships)
updated_content = existing_content + fk_markdown

with open("../schema-reference.md", "w", encoding="utf-8") as f:
    f.write(updated_content)

print(f"Schema file updated with {len(df_relationships)} foreign key relationships")

Schema file updated with 933 foreign key relationships


In [10]:
import yaml


def generate_schema_yaml(df_schema, df_relationships):
    """Convert schema and relationships DataFrames to YAML structure"""

    # Build the schema structure
    schema = {
        "description": f"SQL Server DWH schema. {len(df_schema)} columns across {df_schema['TABLE_NAME'].nunique()} tables.",
        "databases": [],
    }

    # Group by database
    for db_name, db_df in df_schema.groupby("DATABASE_NAME"):
        db_entry = {"name": db_name, "schemas": []}

        # Get relationships for this database
        db_fks = (
            df_relationships[df_relationships["DATABASE_NAME"] == db_name]
            if not df_relationships.empty
            else pd.DataFrame()
        )

        # Group by schema
        for schema_name, schema_df in db_df.groupby("TABLE_SCHEMA"):
            schema_entry = {"name": schema_name, "tables": []}

            # Group by table
            for table_name, table_df in schema_df.groupby("TABLE_NAME"):
                table_entry = {"name": table_name, "columns": []}

                # Add columns
                for _, row in table_df.iterrows():
                    dtype = row["DATA_TYPE"]
                    if (
                        pd.notna(row["CHARACTER_MAXIMUM_LENGTH"])
                        and row["CHARACTER_MAXIMUM_LENGTH"] > 0
                    ):
                        dtype += f"({int(row['CHARACTER_MAXIMUM_LENGTH'])})"
                    elif pd.notna(row["NUMERIC_PRECISION"]):
                        dtype += f"({int(row['NUMERIC_PRECISION'])})"

                    col_entry = {
                        "name": row["COLUMN_NAME"],
                        "type": dtype,
                        "nullable": row["IS_NULLABLE"] == "YES",
                    }
                    table_entry["columns"].append(col_entry)

                # Add foreign keys for this table
                if not db_fks.empty:
                    table_fks = db_fks[
                        (db_fks["ReferencingTableSchema"] == schema_name)
                        & (db_fks["ReferencingTableName"] == table_name)
                    ]
                    if not table_fks.empty:
                        table_entry["foreign_keys"] = []
                        for _, fk_row in table_fks.iterrows():
                            fk_entry = {
                                "name": fk_row["ForeignKeyName"],
                                "column": fk_row["ReferencingColumnName"],
                                "references": {
                                    "schema": fk_row["ReferencedTableSchema"],
                                    "table": fk_row["ReferencedTableName"],
                                    "key": fk_row["ReferencedKeyName"],
                                },
                            }
                            table_entry["foreign_keys"].append(fk_entry)

                schema_entry["tables"].append(table_entry)

            db_entry["schemas"].append(schema_entry)

        schema["databases"].append(db_entry)

    return schema


# Generate YAML
schema_yaml = generate_schema_yaml(df_schema, df_relationships)

# Write to file
with open("../schema-reference.yml", "w", encoding="utf-8") as f:
    yaml.dump(
        schema_yaml,
        f,
        default_flow_style=False,
        allow_unicode=True,
        sort_keys=False,
        width=120,
    )

print(
    f"YAML schema exported: {len(df_schema)} columns, {len(df_relationships)} relationships"
)

YAML schema exported: 12338 columns, 933 relationships
